# Debug MessageStorageHandler Timestamp Issue

This notebook troubleshoots the timestamp mismatch in `test_message_storage.py`. The tests expect `get_formatted_timestamp()` to return a mocked value (`20250414_150000`), but the actual filenames use a real timestamp, indicating the mock is not being applied correctly.

**Steps**:
1. Replicate the test setup for `test_store_messages_no_media`.
2. Inspect the mock for `get_formatted_timestamp()`.
3. Identify why the mock is not being applied.
4. Fix the test to ensure the mock works consistently.

In [1]:
import pytest
import os
import json
from unittest.mock import AsyncMock, MagicMock, patch, mock_open
from datetime import datetime
import pytz
from sqlalchemy.engine import Engine
from helper.message_storage import MessageStorageHandler
from helper.Logger import Logging

# Mock config
config = MagicMock()
config.get_with_default.side_effect = lambda section, key, default: {
    ("message_storage", "temp_dir"): "temp_messages",
    ("message_storage", "media_subdir"): "media",
    ("timezone", "name"): "America/Chicago",
}.get((section, key), default)

# Mock logger
logger = Logging(config, instance_id="debug_message_storage")

# Mock engine
engine = MagicMock(spec=Engine)

# Create MessageStorageHandler instance
message_storage = MessageStorageHandler(config, logger, engine)

# Mock messages
message1 = MagicMock()
message1.id = 1
message1.date = datetime(2025, 4, 14, 15, 0, 0)
message1.text = "Test message 1"
message1.sender_id = 123
message1.reply_to_msg_id = None
message1.media = None
message1.photo = None
message1.video = None
message1.document = None

message2 = MagicMock()
message2.id = 2
message2.date = datetime(2025, 4, 14, 15, 1, 0)
message2.text = "Test message 2"
message2.sender_id = 456
message2.reply_to_msg_id = 1
message2.media = True
message2.photo = True
message2.video = None
message2.document = None

messages = [message1, message2]
channel_id = "12345"
client = AsyncMock()

In [2]:
# Inspect the mock setup without running the full test
with patch("os.makedirs"), \
     patch("os.path.getsize", return_value=1024), \
     patch("helper.timezone.get_formatted_timestamp", return_value="20250414_150000") as mock_timestamp, \
     patch("helper.timezone.to_local_time", side_effect=lambda dt: dt), \
     patch("pytz.timezone", return_value=pytz.timezone("America/Chicago")):
    # Check if the mock is applied
    from helper.timezone import get_formatted_timestamp
    print(f"Mocked timestamp: {get_formatted_timestamp()}")

Mocked timestamp: 20250414_150000


In [3]:
# Run the store_messages method with mocks
with patch("os.makedirs"), \
     patch("os.path.getsize", return_value=1024), \
     patch("helper.timezone.get_formatted_timestamp", return_value="20250414_150000") as mock_timestamp, \
     patch("helper.timezone.to_local_time", side_effect=lambda dt: dt), \
     patch("pytz.timezone", return_value=pytz.timezone("America/Chicago")), \
     patch.object(message_storage.engine, "connect", new_callable=MagicMock) as mock_connect:
    mock_conn = MagicMock()
    mock_connect.return_value.__enter__.return_value = mock_conn
    mock_conn.execute.return_value.fetchone.return_value = None

    mock_file = mock_open()
    with patch("builtins.open", mock_file):
        await message_storage.store_messages(messages, channel_id, client)

    # Inspect the filename
    messages_file = mock_file.call_args_list[0][0][0]
    print(f"Messages file: {messages_file}")
    print(f"Mock calls to get_formatted_timestamp: {mock_timestamp.call_count}")

MyLogger.debug_message_storage - INFO - Stored 2 messages in temp_messages\12345\batch_2025-04-14T16-44-08.606840-05-00.json
MyLogger.debug_message_storage - INFO - Saved batch stats to temp_messages\12345\batch_2025-04-14T16-44-08.606840-05-00_stats.json
MyLogger.debug_message_storage - INFO - Stored batch metadata in channel_fetch_batches for channel 12345
MyLogger.debug_message_storage - INFO - First run: Set earliest ID 1 at 2025-04-14 10:00:00-05:00, latest ID 2 at 2025-04-14 10:01:00-05:00, batch run at 2025-04-14T16-44-08.610341-05-00 for channel 12345


Messages file: temp_messages\12345\batch_2025-04-14T16-44-08.606840-05-00.json
Mock calls to get_formatted_timestamp: 0
